# Combined Facial Similarity Dataset EDA

Comprehensive exploratory analysis across **3 datasets**:
1. **Synthetic** (`subjects_0-1999_72_imgs`) – 2,000 subjects × 72 images
2. **CASIA-WebFace** (`casia_webface_extracted`) – 10,572 subjects, variable images
3. **Olivetti** (`olivetti/`) – 40 subjects × 10 grayscale 64×64 images

**Sections:**
- Dataset summary & scale comparison
- Balance analysis (images per subject)
- Image properties (resolution, brightness, contrast, color mode)
- Intra-subject variation
- Sample visualization grid

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

%matplotlib inline
plt.rcParams.update({'figure.figsize': (14, 6), 'figure.dpi': 120})
sns.set_style('whitegrid')

## 1. Scan & Load All Datasets

In [ ]:
from scripts.eda_combined import scan_all, build_summary

datasets = scan_all()
print(f'Loaded {len(datasets)} datasets: {list(datasets.keys())}')
for k, ds in datasets.items():
    print(f'  {ds.name}: {len(ds.subject_ids)} subjects, {ds.total_images:,} images')

## 2. Dataset Summary

In [ ]:
summary_df = build_summary(datasets)
display(summary_df.style.format({
    'Total Images': '{:,.0f}',
    'Mean Imgs/Subject': '{:.1f}',
    'Median Imgs/Subject': '{:.1f}',
    'Std Imgs/Subject': '{:.2f}',
}).set_caption('Dataset Summary'))

# Grand totals
print(f"\nGrand total: {summary_df['Subjects'].sum():,} subjects, {summary_df['Total Images'].sum():,} images")

## 3. Combined Overview

In [ ]:
from scripts.eda_combined import plot_combined_overview
_ = plot_combined_overview(summary_df)
plt.show()

## 4. Balance Analysis

In [ ]:
from scripts.eda_combined import plot_balance, plot_balance_boxplot
_ = plot_balance(datasets)
plt.show()
_ = plot_balance_boxplot(datasets)
plt.show()

## 5. Image Properties (Sampled)

In [ ]:
from scripts.eda_combined import analyze_image_properties, plot_image_properties

props_df = analyze_image_properties(datasets)
print(f'Sampled {len(props_df)} images for property analysis')
display(props_df.groupby('dataset')[['width', 'height', 'brightness', 'contrast']].describe().round(1))

In [ ]:
_ = plot_image_properties(props_df)
plt.show()

## 6. Intra-Subject Variation

In [ ]:
from scripts.eda_combined import compute_intra_variation, plot_intra_variation

var_df = compute_intra_variation(datasets)
print(f'Computed variation for {len(var_df)} subjects')
display(var_df.groupby('dataset')[['mean_pixel_std', 'max_pixel_std']].describe().round(2))

In [ ]:
_ = plot_intra_variation(var_df)
plt.show()

## 7. Sample Faces Grid

In [ ]:
from scripts.eda_combined import plot_sample_grid
_ = plot_sample_grid(datasets, n_subjects=3, n_imgs=6)
plt.show()

## 8. Key Findings & Observations

Run this cell after executing all above to auto-generate findings.

In [ ]:
findings = []

# Balance
for _, row in summary_df.iterrows():
    if row['Std Imgs/Subject'] < 1:
        findings.append(f"✅ **{row['Dataset']}** is perfectly balanced ({int(row['Mean Imgs/Subject'])} imgs/subject)")
    else:
        findings.append(f"⚠️ **{row['Dataset']}** is imbalanced (std={row['Std Imgs/Subject']:.1f}, range={int(row['Min Imgs/Subject'])}-{int(row['Max Imgs/Subject'])})")

# Scale
total_subj = summary_df['Subjects'].sum()
total_imgs = summary_df['Total Images'].sum()
findings.append(f"📊 Combined: **{total_subj:,} subjects**, **{total_imgs:,} images**")

# Resolution diversity
if len(props_df) > 0:
    n_resolutions = props_df.groupby('dataset').apply(lambda g: g[['width','height']].drop_duplicates().shape[0])
    for ds, n_res in n_resolutions.items():
        if n_res == 1:
            w = props_df[props_df['dataset']==ds]['width'].iloc[0]
            h = props_df[props_df['dataset']==ds]['height'].iloc[0]
            findings.append(f"📐 **{ds}** has uniform resolution: {w}×{h}")
        else:
            findings.append(f"📐 **{ds}** has {n_res} distinct resolutions")

display(Markdown('### Automated Findings\n\n' + '\n'.join(f'- {f}' for f in findings)))